# Resolving Conflicts by Encoding the Precedences into the Grammar

The notebook [01-Conflicts.ipynb](01-Conflicts.ipynb) has shown that the following grammar is *ambiguous*,
because it does not specify the precedences of the arithmetical operators:
```
    expr : expr "+" expr
         | expr "-" expr
         | expr "*" expr
         | expr "/" expr
         | expr "^" expr
         | "(" expr ")"
         | NUMBER
```

The parser generators of the `yacc` family, among them `Ply`, resolve the resulting *shift/reduce conflicts*
with *operator precedence declarations* of the form
```
    precedence = ( ('left' , '+', '-'),   # precedence 1
                   ('left' , '*', '/'),   # precedence 2
                   ('right', '^')         # precedence 3
                 )
```

<b style="color:red">`Larkita` does not support operator precedence declarations.</b>
There is no way to attach a precedence or an associativity to a terminal, and there is no way to influence
the resolution of an individual shift/reduce conflict: `Larkita` resolves *every* shift/reduce conflict in
favour of shifting.

Therefore, the precedences and associativities have to be encoded into the grammar itself.

## Encoding the Precedences

The recipe is the one that we have already used in the chapter on context-free languages:  we introduce one
syntactical variable $v_n$ for every precedence level $n$ and define the variable of a given precedence level
in terms of the variable of the next higher precedence level.  Let $o$ be an operator of precedence level $n$.

* If $o$ is **left associative**, the corresponding rule is **left recursive**:
  $$ v_n \rightarrow v_n \; o \; v_{n+1} $$
* If $o$ is **right associative**, the rule is **right recursive**:
  $$ v_n \rightarrow v_{n+1} \; o \; v_n $$
* If $o$ is **non-associative**, the rule uses neither recursion:
  $$ v_n \rightarrow v_{n+1} \; o \; v_{n+1} $$

For the grammar above we need three precedence levels plus one variable for the expressions that bind most
tightly:

| variable | precedence | operators | associativity |
|:---------|:-----------|:----------|:--------------|
| `expr`   | 1          | `+`, `-`  | left          |
| `prod`   | 2          | `*`, `/`  | left          |
| `fact`   | 3          | `^`       | right         |
| `atom`   | &mdash;    | &mdash;   | &mdash;       |

Note how the associativities show up in the grammar below:  the rules for `expr` and `prod` are *left*
recursive, while the rule `fact : atom "^" fact` is *right* recursive.

The abstract syntax tree that `Larkita` builds is determined by the grammar:  The string literals of an alternative
form the *head* of the tuple that is returned for this alternative, while all other symbols become its children.

1. Hence the alternative `expr "+" prod` yields the tuple `('+', lhs, rhs)`, so that we need no aliases.
2. An alternative that consists of a single symbol, like `expr : prod`, returns the abstract syntax tree of this
   symbol.  Without this, every number would be buried under a chain `expr` &rarr; `prod` &rarr; `fact` &rarr; `atom`
   that carries no information.
3. The annotation `%ignore "(", ")"` at the end of the alternative `"(" expr ")"` removes the parentheses from its
   head.  Then this alternative consists of a single symbol and returns the abstract syntax tree of `expr`.

None of these features has any influence on the parse table:  they can neither introduce nor remove a conflict.

In [ ]:
grammar = r"""
    expr : expr "+" prod
         | expr "-" prod
         | prod

    prod : prod "*" fact
         | prod "/" fact
         | fact

    fact : atom "^" fact
         | atom

    atom : "(" expr ")"    %ignore "(", ")"
         | NUMBER

    NUMBER : /0|[1-9][0-9]*/

    %import common.WS
    %ignore WS
"""

## Specification of the Parser

We turn the `ConflictWarning` that `Larkita` issues for a grammar with shift-reduce conflicts into an exception.
Then a conflict cannot be overlooked.  Reduce-reduce conflicts raise an exception of class `GrammarError` anyway.

Since no exception is raised below, our grammar is an `LR(1)` grammar.

In [ ]:
import warnings

from larkita import Larkita, ConflictWarning

with warnings.catch_warnings():
    warnings.simplefilter('error', ConflictWarning)
    parser = Larkita(grammar, out_file='conflicts-resolved.out')

## Inspecting the LR(1) States

`Ply` writes the LALR states into the file `parser.out`.  `Larkita` writes the states of the canonical LR(1)
automaton into the file given by the keyword argument `out_file`.  Every state lists its *marked rules* together with
their *lookahead* tokens, followed by the actions and the gotos of the state.  The marked rules that are added by the
closure are marked with a `+`.

`Larkita` marks the position of the parser inside a rule with the bullet `•` that we use in the lecture notes, and
the end of the input is called `$END`.

Note that there is no state that offers both a `shift` and a `reduce` action for the same token:  all
conflicts are gone.

In [ ]:
with open('conflicts-resolved.out') as handle:
    print(handle.read())

## Building an Abstract Syntax Tree

The parser generated by `Larkita` returns the abstract syntax tree as a nested tuple.  Hence we can use the function
`tuple2dot` directly in order to display it.

In [ ]:
%run ../AST2Dot.ipynb

In [ ]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:  # declarations for the type checker, never executed
    import graphviz as gv
    type AST = str | int | float | tuple[str, *tuple["AST", ...]]
    def tuple2dot(t: AST) -> gv.Digraph: ...

The function `test(s)` takes a string `s` as its argument and tries to parse this string.  If all goes well,
an abstract syntax tree is returned and displayed.  If the string can't be parsed, `Larkita` raises an exception
of class `UnexpectedInput`.  The message of this exception already contains the line, the column and the
tokens that would have been acceptable, so there is no need for a function like `p_error`.

In [ ]:
from larkita import UnexpectedInput

def test(s):
    try:
        t = parser.parse(s)
    except UnexpectedInput as e:
        print(f'Syntax error:\n{e}')
        return None
    d = tuple2dot(t)
    display(d)
    return t

In [ ]:
test('2^3^4*5+6-7/8^9')

In [ ]:
test('1+2*3^4')

In [ ]:
test('1 * 2 + (3^4)^5')

The next two examples check the associativities:  the additive operators group to the left, while the
exponentiation operator groups to the right.

In [ ]:
test('1-2-3')

In [ ]:
test('2^3^4')

Finally, an example of a string that cannot be parsed.

In [ ]:
test('1 + * 2')